<a href="https://colab.research.google.com/github/freezetea/data-science-2026/blob/main/Pertemuan10_%5BAlif_Firdaus_Hidayatullah%5D_%5B250401020166%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pertemuan 10
Nama : Alif Firdaus Hidayatullah
NIM  : 250401020166
Kelas : IF405

**Muat dan Eksplorasi Data**

In [24]:
import pandas as pd

# Load dataset Telco Churn
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()
print('Shape data:', df.shape)
print('\nProporsi kelas Churn:')
print(df["Churn"].value_counts(normalize=True).round(3))
print(df["Churn"].value_counts())

Shape data: (7043, 21)

Proporsi kelas Churn:
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64
Churn
No     5174
Yes    1869
Name: count, dtype: int64


**Preprocessing**

In [25]:
from sklearn.model_selection import train_test_split

# Drop customerID karena tidak relevan
df = df.drop('customerID', axis=1, errors='ignore')

# Tangani nilai kosong pada TotalCharges (ganti spasi dengan NaN lalu imputasi dengan median)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.replace(' ', '', regex=False), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Map target Churn dari 'Yes'/'No' ke 1/0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Pisahkan fitur dan target
X = df.drop('Churn', axis=1)
y = df['Churn']

# Terapkan One-Hot Encoding pada fitur kategorikal
X = pd.get_dummies(X, drop_first=True, dtype=int)

# Train-Test Split 80:20 secara stratified
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"X_tr shape: {X_tr.shape}, X_te shape: {X_te.shape}")

X_tr shape: (5634, 30), X_te shape: (1409, 30)


**Latih Model**

In [26]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_tr, y_tr)

print("Model berhasil dilatih.")

Model berhasil dilatih.


**Evaluasi Model**

In [27]:
from sklearn.metrics import classification_report, roc_auc_score

# Prediksi kelas
y_pred = rf.predict(X_te)

# Prediksi probabilitas
y_prob = rf.predict_proba(X_te)[:, 1]

# Classification Report
print(classification_report(
    y_te,
    y_pred,
    target_names=["Tidak Churn","Churn"]
))

# ROC-AUC
roc = roc_auc_score(y_te, y_prob)

print("ROC-AUC Score :", roc)

              precision    recall  f1-score   support

 Tidak Churn       0.83      0.89      0.86      1035
       Churn       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC Score : 0.8246208891988943


**Prediksi Probabilitas dan Simpulkan**

In [28]:
# Menghitung probabilitas churn
probabilitas = rf.predict_proba(X_te)[:, 1]

# Menampilkan hasil prediksi
hasil = pd.DataFrame({
    "Aktual": y_te.values,
    "Probabilitas Churn": probabilitas
})

hasil = hasil.sort_values(
    by="Probabilitas Churn",
    ascending=False
)

hasil.head(10)

,Aktual,Probabilitas Churn
1289,1,1.000000
171,1,0.993333
341,1,0.990000
618,1,0.990000
1252,1,0.986667
629,0,0.986667
889,0,0.970000
1178,1,0.963333
1109,1,0.960000
788,1,0.950000


**Kesimpulan**

Pada praktikum ini saya mempelajari cara membangun model klasifikasi menggunakan algoritma Random Forest pada dataset Telco Customer Churn. Dataset yang digunakan memiliki distribusi kelas yang tidak seimbang, sehingga digunakan parameter class_weight="balanced" untuk membantu model mengenali kelas churn dengan lebih baik.

Berdasarkan hasil evaluasi, model memberikan performa yang cukup baik dengan nilai Recall, F1-score, dan ROC-AUC yang stabil. Praktikum ini juga memberikan pemahaman bahwa kualitas data serta pemilihan metode evaluasi memiliki pengaruh yang besar terhadap hasil prediksi.